In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import torch.nn.functional as F

class BuggyAttention(nn.Module):
    def __init__(self, embed_dim, n_head):
        super(BuggyAttention, self).__init__()
        self.embed_dim = embed_dim
        self.n_head = n_head
        self.head_dim = embed_dim // n_head
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x, mask=None):
        batch_size, seq_len, emb_dim = x.size()
        if emb_dim != self.embed_dim:
            raise ValueError("embed dim not match")
        qkv = self.qkv(x).reshape(batch_size, seq_len, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn_weight = torch.matmul(q, k.transpose(-2, -1))
        if mask is not None:
            attn_weight = attn_weight + mask
        attn_weight = F.softmax(attn_weight, dim=-1)
        output = torch.matmul(attn_weight, v)
        output = output.transpose(1, 2).reshape(batch_size, seq_len, self.embed_dim)
        output = self.out_proj(output)
        return output

class Attention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.1):
        super(Attention, self).__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.head_dim = d_model // n_head
        if d_model % n_head != 0:
            raise ValueError("embed_dim % n_head not match")
        # attention scaling
        self.scaling = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None, return_attn=False):
        batch, seq, emb = x.size()
        if emb != self.d_model:
            raise ValueError("emb not match")
        qkv = self.qkv(x).reshape(batch, seq, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scaling
        # Apply mask if provided
        if mask is not None:
            # THE FIX: Properly handle different mask dimensions
            if mask.dim() == 2:
                # For causal mask: (seq_len, seq_len) -> (1, 1, seq_len, seq_len)
                mask = mask.unsqueeze(0).unsqueeze(0)
            elif mask.dim() == 3:
                # For padding mask: (batch_size, seq_len, seq_len) -> (batch_size, 1, seq_len, seq_len)
                mask = mask.unsqueeze(1)
            
            # Ensure mask is bool type and properly expanded
            if mask.dtype != torch.bool:
                mask = mask.bool()
                
            # Use masked_fill_ to set masked positions to -inf
            attn = attn.masked_fill(mask, float('-inf'))
        attn = self.dropout(F.softmax(attn, dim=-1))
        out = self.out_proj(torch.matmul(attn, v).transpose(1, 2).reshape(batch, seq, self.d_model))
        if return_attn:
            return out, attn
        return out


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5_000):
        super(PositionalEncoding, self).__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10_000.)/100.))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        # We use register_buffer instead of register_parameter for positional encodings because:
        # 1. These values are fixed and should not be updated during training
        # 2. They are deterministic based on position and dimension, not learned
        # 3. We still want them to persist in the state_dict and move to GPU with the model
        # 4. Unlike parameters, buffers don't accumulate gradients during backpropagation
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        return x + self.pe

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5_000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        # unsqueeze(1) adds a dimension at index 1, transforming shape from [max_len] to [max_len, 1]
        # This is needed to properly broadcast when multiplying with div_term later
        pos = torch.arange(max_len).unsqueeze(1)
        mul = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10_000.)/d_model))
        pe = torch.zeros(max_len, d_model)
        # pos shape: [max_len, 1]
        # mul shape: [d_model//2]
        # When multiplied together, pos * mul will broadcast to shape [max_len, d_model//2]
        # This matches the needed shape for assigning to pe[:, 0::2] and pe[:, 1::2]
        pe[:, 0::2] = torch.sin(pos * mul)
        pe[:, 1::2] = torch.cos(pos * mul)
        self.register_buffer('pe', pe)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # unsqueeze(0) adds a new dimension at the start (dimension 0)
        # For example, if pe shape is [seq_len, d_model]
        # after unsqueeze(0) it becomes [1, seq_len, d_model]
        # This allows broadcasting when adding to x which has shape [batch, seq_len, d_model]
        return self.dropout(x + self.pe[:x.size(-2)].unsqueeze(0))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForward, self).__init__()
        self.l1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.l2 = nn.Linear(d_ff, d_model)
        
    def forward(self, x):
        return self.l2(self.dropout(self.l1(x)))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForward, self).__init__()
        self.l1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(d_ff, d_model)
        
    def forward(self, x):
        return self.l2(self.dropout(self.relu(self.l1(x))))

class BuggyTransformerLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super(BuggyTransformerLayer, self).__init__()
        self.self_attn = Attention(d_model, n_head)
        self.ffd = FeedForward(d_model, d_ff)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, src, src_mask=None):
        src = src + self.dropout(self.self_attn(src, src_mask))
        src = src + self.dropout(self.ffd(src))
        return src

class TransformerLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super(TransformerLayer, self).__init__()
        self.self_attn = Attention(d_model, n_head)
        self.ffd = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x, mask=None):
        x = x + self.dropout(self.self_attn(self.norm1(x), mask))
        x = x + self.dropout(self.ffd(self.norm2(x)))
        return x
    
# Testing code to verify fixes
def test_model_fixes():
    # Create sample input
    batch_size, seq_len, d_model, n_head, d_ff = 2, 10, 512, 8, 2048
    x = torch.randn(batch_size, seq_len, d_model)
    mask = torch.zeros(seq_len, seq_len).bool()
    
    # Test attention modules
    buggy_attn = BuggyAttention(d_model, n_head)
    fixed_attn = Attention(d_model, n_head)
    
    try:
        # This might fail due to bugs
        buggy_out = buggy_attn(x, mask)
        print("Buggy attention output shape:", buggy_out.shape)
    except Exception as e:
        print(f"Buggy attention error: {e}")
    
    # This should work
    fixed_out = fixed_attn(x, mask)
    print("Fixed attention output shape:", fixed_out.shape)
    
    # Test full transformer layers
    buggy_layer = BuggyTransformerLayer(d_model, n_head, d_ff)
    fixed_layer = TransformerLayer(d_model, n_head, d_ff)
    
    try:
        buggy_layer_out = buggy_layer(x, mask)
        print("Buggy layer output shape:", buggy_layer_out.shape)
    except Exception as e:
        print(f"Buggy layer error: {e}")
    
    fixed_layer_out = fixed_layer(x, mask)
    print("Fixed layer output shape:", fixed_layer_out.shape)


# Testing code with proper mask creation
def test_model_fixes2():
    # Create sample input
    batch_size, seq_len, d_model = 2, 10, 512
    x = torch.randn(batch_size, seq_len, d_model)
    
    # Create a proper attention mask (where True values are positions to be masked)
    # This simulates a padding mask where the last 3 tokens are padding
    mask = torch.zeros(batch_size, seq_len, seq_len, dtype=torch.bool)
    for i in range(batch_size):
        mask[i, :, 7:] = True  # Mask out the last 3 positions
    
    # Test attention module
    fixed_attn = Attention(d_model, 8)
    
    # This should work
    fixed_out = fixed_attn(x, mask)
    print("Fixed attention output shape:", fixed_out.shape)
    
    # Test full transformer layer
    fixed_layer = TransformerLayer(d_model, 8, 2048)
    
    fixed_layer_out = fixed_layer(x, mask)
    print("Fixed layer output shape:", fixed_layer_out.shape)
    
    # Test with causal mask (for decoder)
    causal_mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1)
    fixed_out_causal = fixed_attn(x, causal_mask)
    print("Fixed attention with causal mask output shape:", fixed_out_causal.shape)
    
    # Validate the outputs are different with different masks
    print("Are outputs with different masks different?", 
          not torch.allclose(fixed_out, fixed_out_causal))

# Run the test
test_model_fixes()
test_model_fixes2()
        

Buggy attention output shape: torch.Size([2, 10, 512])
Fixed attention output shape: torch.Size([2, 10, 512])
Buggy layer output shape: torch.Size([2, 10, 512])
Fixed layer output shape: torch.Size([2, 10, 512])
Fixed attention output shape: torch.Size([2, 10, 512])
Fixed layer output shape: torch.Size([2, 10, 512])
Fixed attention with causal mask output shape: torch.Size([2, 10, 512])
Are outputs with different masks different? True


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.1) -> None:
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.scale = self.head_dim ** -0.5
        self.qw = nn.Linear(d_model, d_model)
        self.kw = nn.Linear(d_model, d_model)
        self.vw = nn.Linear(d_model, d_model)
        self.ow = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, q, k, v, mask=None, return_attn=False):
        batch = q.size(0)
        q_l, k_l, v_l = q.size(1), k.size(1), v.size(1)
        
        # Reshape q,k,v to split heads and transpose for batch matrix multiply
        q = self.qw(q).view(batch, q_l, self.n_head, self.head_dim).transpose(1, 2)
        k = self.kw(k).view(batch, k_l, self.n_head, self.head_dim).transpose(1, 2)
        v = self.vw(v).view(batch, v_l, self.n_head, self.head_dim).transpose(1, 2)
        
        attn_score = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        
        if mask is not None:
            print(mask.size())
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            if mask.dtype != torch.bool:
                mask = mask.bool()
            # Expand mask to match attention score dimensions
            
            print(mask.size())
            print(attn_score.size())
            mask = mask.expand(batch, self.n_head, q_l, k_l)
            attn_score = attn_score.masked_fill(mask, float('-inf'))
            
        attn = F.softmax(attn_score, dim=-1)
        attn = self.dropout(attn)  # Apply dropout after softmax but before matmul
        out = torch.matmul(attn, v)
        # Transpose and reshape back
        out = out.transpose(1, 2).contiguous().view(batch, q_l, self.d_model)
        out = self.ow(out)
        
        if return_attn:
            return out, attn
        return out
        

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, src, src_mask=None):
        # Self-attention block with pre-normalization
        src2 = self.norm1(src)
        src2 = self.self_attn(src2, src2, src2, src_mask)
        src = src + self.dropout1(src2)
        
        # Feed-forward block with pre-normalization
        src2 = self.norm2(src)
        src2 = self.feed_forward(src2)
        src = src + self.dropout2(src2)
        return src

class Encoder(nn.Module):
    def __init__(self, d_model, n_head, d_ff, n_layer, dropout=0.1) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_head, d_ff, dropout)
            for _ in range(n_layer)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, src, mask=None):
        output = src
        for layer in self.layers:
            output = layer(output, mask)
        return self.norm(output)


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # First self-attention layer is for masked self-attention
        # This prevents the decoder from looking at future tokens during training
        # Essential for autoregressive generation
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Cross-attention layer attends to encoder outputs
        # This allows decoder to use information from the entire input sequence
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, tgt, context, tgt_causal_mask=None, tgt_pad_mask=None, context_pad_mask=None):
        # First self-attention block with causal masking
        # This ensures decoder can only attend to previous positions
        # Critical for maintaining autoregressive property
        tgt2 = self.norm1(tgt)
        
        # Apply both causal and padding masks for self-attention
        # First apply causal mask (all future positions), then pad mask (padding tokens)
        tgt2 = self.self_attn(tgt2, tgt2, tgt2, tgt_causal_mask)
        
        # If we have a padding mask, apply it separately in a second attention layer
        # This is one approach to combine two different masks
        if tgt_pad_mask is not None:
            tgt2 = self.self_attn(tgt2, tgt2, tgt2, tgt_pad_mask)
            
        tgt = tgt + self.dropout1(tgt2)
        
        # Second attention block - cross-attention to encoder outputs
        # This allows decoder to access the full input sequence
        # No causal mask needed here since we want to see all encoder outputs
        tgt2 = self.norm2(tgt)
        tgt2 = self.cross_attn(tgt2, context, context, context_pad_mask)
        tgt = tgt + self.dropout2(tgt2)
        
        # Feed-forward block
        tgt2 = self.norm3(tgt)
        tgt2 = self.feed_forward(tgt2)
        tgt = tgt + self.dropout3(tgt2)
        
        return tgt


class Decoder(nn.Module):
    def __init__(self, d_model, n_head, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_head, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, tgt, memory, tgt_causal_mask=None, tgt_pad_mask=None, memory_pad_mask=None):
        output = tgt
        for layer in self.layers:
            output = layer(output, memory, tgt_causal_mask, tgt_pad_mask, memory_pad_mask)
        return self.norm(output)


class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, n_head, d_ff, n_layer, dropout=0.1, pad_idx=0, max_len=50_000):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=pad_idx)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.encoder = Encoder(d_model, n_head, d_ff, n_layer, dropout)
        self.decoder = Decoder(d_model, n_head, d_ff, n_layer, dropout)
        self.output_layer = nn.Linear(d_model, tgt_vocab)
        self._init_parameters()
        
    def _init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, src, tgt):
        src_pad_mask = self._make_pad_mask(src)
        tgt_pad_mask = self._make_pad_mask(tgt)
        tgt_causal_mask = self._make_causal_mask(tgt)
        src = self.src_emb(src) * math.sqrt(self.d_model)
        src = self.pos_encoding(src)
        tgt = self.tgt_emb(tgt) * math.sqrt(self.d_model)
        tgt = self.pos_encoding(tgt)
        
        context = self.encoder(src, src_pad_mask)
        output = self.decoder(tgt, context, tgt_causal_mask, tgt_pad_mask, src_pad_mask)
        return self.output_layer(output)
    
    def _make_causal_mask(self, x):
        seq = x.size(1)
        mask = torch.triu(torch.ones(seq, seq), diagonal=1).bool()
        return mask

    def _make_pad_mask(self, x):
        return (x == self.pad_idx).unsqueeze(1)

def test_transformer_masks():
    # Test parameters
    batch_size, seq_len = 2, 10
    src_vocab_size, tgt_vocab_size = 1000, 1000
    d_model, num_heads = 512, 8
    d_ff, num_layers = 2048, 2  # Use fewer layers for testing
    pad_idx = 0
    
    # Create input with padding
    src = torch.randint(1, src_vocab_size, (batch_size, seq_len))
    tgt = torch.randint(1, tgt_vocab_size, (batch_size, seq_len))
    
    print(src.size())
    print(tgt.size())
    
    # Add padding tokens to test padding mask
    src[0, -2:] = pad_idx  # Pad last 2 positions of first batch
    tgt[1, -3:] = pad_idx  # Pad last 3 positions of second batch
    
    # Create transformer with fixed mask handling
    transformer = Transformer(
        src_vocab_size, tgt_vocab_size, d_model, num_heads,
        d_ff, num_layers, pad_idx=pad_idx
    )
    
    # Test mask creation
    src_pad_mask = transformer._make_pad_mask(src)
    tgt_pad_mask = transformer._make_pad_mask(tgt)
    tgt_causal_mask = transformer._make_causal_mask(tgt)
    
    print(f"Source padding mask shape: {src_pad_mask.shape}")
    print("Sample source padding mask:")
    print(src_pad_mask[0, 0, -5:])  # Last 5 positions of first batch
    
    print(f"\nTarget padding mask shape: {tgt_pad_mask.shape}")
    print("Sample target padding mask:")
    print(tgt_pad_mask[1, 0, -5:])  # Last 5 positions of second batch
    
    print(f"\nCausal mask shape: {tgt_causal_mask.shape}")
    print("Sample causal mask (top-left corner):")
    print(tgt_causal_mask[:5, :5])
    
    # Test forward pass with fixed masks
    output = transformer(src, tgt[:, :-1])
    print(f"\nTransformer output shape: {output.shape}")
    
    # Test that the output is valid (no NaNs)
    print("Output contains NaN:", torch.isnan(output).any().item())
    
    print("\nTransformer with fixed masks tested successfully!")

# Execute test
test_transformer_masks()

torch.Size([2, 10])
torch.Size([2, 10])
Source padding mask shape: torch.Size([2, 1, 10])
Sample source padding mask:
tensor([False, False, False,  True,  True])

Target padding mask shape: torch.Size([2, 1, 10])
Sample target padding mask:
tensor([False, False,  True,  True,  True])

Causal mask shape: torch.Size([10, 10])
Sample causal mask (top-left corner):
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
torch.Size([2, 1, 10])
torch.Size([2, 1, 1, 10])
torch.Size([2, 8, 10, 10])
torch.Size([2, 1, 10])
torch.Size([2, 1, 1, 10])
torch.Size([2, 8, 10, 10])
torch.Size([9, 9])
torch.Size([1, 1, 9, 9])
torch.Size([2, 8, 9, 9])
torch.Size([2, 1, 9])
torch.Size([2, 1, 1, 9])
torch.Size([2, 8, 9, 9])
torch.Size([2, 1, 10])
torch.Size([2, 1, 1, 10])
torch.Size([2, 8, 9, 10])
torch.Size([9, 9])
torch.Size([1, 1, 9, 9]

In [7]:
def test2():
    att = Transformer(
        src_vocab=100, tgt_vocab=200, pad_idx=0, d_model=512, n_layer=6, n_head=8, 
        d_ff=1024, dropout=0.1)
    x = torch.randint(0, 100, (4, 64))
    y = torch.randint(0, 200, (4, 64))
    out = att(x, y)
    print(out.shape)
test2()

torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([64, 64])
torch.Size([1, 1, 64, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([64, 64])
torch.Size([1, 1, 64, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 1, 64])
torch.Size([4, 1, 1, 64])
torch.Size([4, 8, 64, 64])
torch.Size([64, 64])
torch.Size([1, 1, 64, 64])
torch.Size([4, 8, 64, 64])
torch.Size([4, 